# 11b | ML Algorithms from Scratch

Coding questions for ML roles and some DS roles are not LeetCode but rather: "**Implement k-means in NumPy without sklearn** (or kNN, gradient descent for linear regression, logistic regression, AUC)."

These problems aren't hard, but they require both understanding the algorithm and fluency in NumPy. This lesson writes out the 7 most commonly asked ones. Each is only about ten lines.

About 80 minutes.

### ✏️ Exercise 1 (write it yourself)

**1. Train/test split**

Write a function `split_indices(n, test_frac, rng)`: shuffle the indices 0 to n−1, take the first `int(n * test_frac)` as the test set and the rest as the training set. Return `(train_idx, test_idx)` as two NumPy arrays.

`rng.permutation(n)` returns a random permutation of 0 to n−1.

In [ ]:
import numpy as np

def split_indices(n, test_frac, rng):
    # Write your code below
    pass

<details>
<summary><b>Answer</b></summary>

```python
import numpy as np

def split_indices(n, test_frac, rng):
    idx = rng.permutation(n)
    n_test = int(n * test_frac)
    return idx[n_test:], idx[:n_test]
```

</details>

### ✏️ Exercise 2 (write it yourself)

**2. Standardization (without leakage)**

Write a function `standardize(X_train, X_test)`: standardize both the training and test sets using the mean and standard deviation of each column of the **training set**. Return `(X_train_s, X_test_s)`.

In [ ]:
def standardize(X_train, X_test):
    # Write your code below
    pass

<details>
<summary><b>Answer</b></summary>

```python
def standardize(X_train, X_test):
    mu = X_train.mean(axis=0)
    sd = X_train.std(axis=0)
    return (X_train - mu) / sd, (X_test - mu) / sd
```

</details>

## 3. AUC

Implement it from the definition: **AUC = the probability that a randomly chosen positive scores higher than a randomly chosen negative**; ties count as 0.5.

The most direct approach enumerates all (positive, negative) pairs. NumPy broadcasting avoids loops: `pos[:, None]` turns the positive scores into a column, which is compared with a row of negative scores `neg[None, :]`, giving a "number of positives × number of negatives" matrix.

### ✏️ Exercise 3 (write it yourself)

Write a function `auc_score(y, s)`: `y` holds 0/1 labels and `s` the model scores. Return the AUC according to the definition above.

In [ ]:
def auc_score(y, s):
    pos = s[y == 1]
    neg = s[y == 0]
    # Write your code below
    pass

<details>
<summary><b>Hint</b></summary>

`(pos[:, None] > neg[None, :]).sum()` is the number of pairs where the positive scores higher; with `==` it is the number of tied pairs. Result = (pairs higher + 0.5 × tied pairs) ÷ total pairs. A double for loop works too.

</details>

<details>
<summary><b>Answer</b></summary>

```python
def auc_score(y, s):
    pos = s[y == 1]
    neg = s[y == 0]
    greater = (pos[:, None] > neg[None, :]).sum()
    ties = (pos[:, None] == neg[None, :]).sum()
    return (greater + 0.5 * ties) / (len(pos) * len(neg))
```

This approach is O(#positives × #negatives). For large data, use a sorting approach (sort by score and compute from rank sums, O(n log n)); mentioning it in the interview is enough.

</details>

## 4. kNN

For a new point `x`:

1. Compute its Euclidean distance to every training point: `np.sqrt(((X_train - x) ** 2).sum(axis=1))`;
2. Find the indices of the k smallest distances: `np.argsort(dist)[:k]` (`argsort` returns "the original indices of the elements in sorted order");
3. The most frequent label among these k points is the prediction.

### ✏️ Exercise 4 (write it yourself)

Write a function `knn_predict(X_train, y_train, x, k)` that returns the predicted class for a single point `x`. Labels are nonnegative integers; `np.bincount(labels).argmax()` gives the most frequent label.

In [ ]:
def knn_predict(X_train, y_train, x, k):
    # Write your code below
    pass

<details>
<summary><b>Answer</b></summary>

```python
def knn_predict(X_train, y_train, x, k):
    dist = np.sqrt(((X_train - x) ** 2).sum(axis=1))
    nearest = np.argsort(dist)[:k]
    return np.bincount(y_train[nearest]).argmax()
```

</details>

## 5. k-means

Repeat two steps until convergence (simplified here to a fixed number of iterations):

1. **Assign:** assign each point to its nearest center;
2. **Update:** move each center to the mean of the points assigned to it.

Compute all point-to-center distances with broadcasting as well: `X[:, None, :] - centers[None, :, :]` has shape (points, k, dimensions); summing squares over the last axis gives a (points, k) distance matrix, and `argmin(axis=1)` is the index of each point's nearest center.

### ✏️ Exercise 5 (write it yourself)

Write a function `kmeans(X, init_centers, n_iter)`: starting from the given initial centers, iterate `n_iter` times and return `(centers, labels)`. `labels` is each point's final cluster index (corresponding to the returned `centers`). You don't need to handle empty clusters.

In [ ]:
def kmeans(X, init_centers, n_iter):
    centers = init_centers.copy()
    k = len(centers)
    for _ in range(n_iter):
        # 1. Assign: find each point's nearest center to get labels
        # 2. Update: centers[j] = mean of the points in cluster j
        pass
    return centers, labels

<details>
<summary><b>Hint</b></summary>

Assign: `dist = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)`, `labels = dist.argmin(axis=1)`. Update: `centers = np.array([X[labels == j].mean(axis=0) for j in range(k)])`.

</details>

<details>
<summary><b>Answer</b></summary>

```python
def kmeans(X, init_centers, n_iter):
    centers = init_centers.copy()
    k = len(centers)
    for _ in range(n_iter):
        dist = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)     # (points, k)
        labels = dist.argmin(axis=1)
        centers = np.array([X[labels == j].mean(axis=0) for j in range(k)])
    return centers, labels
```

Points to add in an interview: a real implementation needs a convergence check (centers stop moving), handling of empty clusters, and k-means++ initialization with multiple runs, keeping the best result.

</details>

## 6. Gradient descent: linear regression

The loss function is $L(w) = \frac{1}{n}\|Xw - y\|^2$, and its gradient is $\frac{2}{n}X^\top(Xw - y)$.

```
w = all zeros
repeat n_iter times:
    grad = (2 / n) * X.T @ (X @ w - y)
    w = w - lr * grad
```

The first column of `X` is all ones (the intercept). Standardize the features first; otherwise the gradients differ greatly across directions and it is hard to choose a learning rate.

### ✏️ Exercise 6 (write it yourself)

Write a function `gd_linear(X, y, lr, n_iter)` that runs gradient descent starting from an all-zero `w` and returns `w`.

In [ ]:
def gd_linear(X, y, lr, n_iter):
    n, d = X.shape
    w = np.zeros(d)
    # Write your code below
    pass

<details>
<summary><b>Answer</b></summary>

```python
def gd_linear(X, y, lr, n_iter):
    n, d = X.shape
    w = np.zeros(d)
    for _ in range(n_iter):
        grad = (2 / n) * X.T @ (X @ w - y)
        w = w - lr * grad
    return w
```

</details>

## 7. Gradient descent: logistic regression

It differs from linear regression only by a sigmoid. The gradient of the mean log loss is:

$$\nabla L(w) = \frac{1}{n}X^\top\big(\sigma(Xw) - y\big)$$

The form is almost identical to linear regression: "prediction minus truth," multiplied by $X^\top$.

### ✏️ Exercise 7 (write it yourself)

Write a function `fit_logistic(X, y, lr, n_iter)` that starts from an all-zero `w`, minimizes the mean log loss with gradient descent, and returns `w`. The first column of `X` is the intercept.

In [ ]:
def fit_logistic(X, y, lr, n_iter):
    n, d = X.shape
    w = np.zeros(d)
    # Write your code below
    pass

<details>
<summary><b>Hint</b></summary>

Three lines inside the loop: `p = 1 / (1 + np.exp(-(X @ w)))`; `grad = X.T @ (p - y) / n`; `w = w - lr * grad`.

</details>

<details>
<summary><b>Answer</b></summary>

```python
def fit_logistic(X, y, lr, n_iter):
    n, d = X.shape
    w = np.zeros(d)
    for _ in range(n_iter):
        p = 1 / (1 + np.exp(-(X @ w)))
        grad = X.T @ (p - y) / n
        w = w - lr * grad
    return w
```

Possible follow-ups: with L2 regularization the gradient gains a `lam * w` term (the intercept is usually not penalized); at prediction time `p >= 0.5` is classified as positive; a real implementation must guard against `exp` overflow.

</details>

## Summary

| Algorithm | The core line or two |
|---|---|
| AUC | pairs with positive score > negative score ÷ total pairs |
| kNN | `np.argsort(dist)[:k]` → majority vote |
| k-means | `dist.argmin(axis=1)` → mean of each cluster |
| Linear regression GD | `w -= lr * (2/n) * X.T @ (X @ w - y)` |
| logistic GD | `w -= lr * X.T @ (sigmoid(X @ w) - y) / n` |

Rewrite these a week later without looking at the answers. While writing, also practice **talking as you code**: what each step does, its complexity, and what else a real implementation would need to consider.

Next: **12**.